In [25]:
#! pip install kafka-python

In [26]:
from kafka import KafkaAdminClient

In [27]:
# create connection to kafka broker 

admin= KafkaAdminClient(
    bootstrap_servers="localhost:9092"
)
print("connected successfully")

connected successfully


In [28]:
#list all topics 
topics = admin.list_topics()
print(topics)

['sensor-telemetry']


In [29]:
admin.describe_cluster()


{'throttle_time_ms': 0,
 'brokers': [{'node_id': 1, 'host': 'localhost', 'port': 9092, 'rack': None}],
 'cluster_id': '5L6g3nShT-eMCtK--X86sw',
 'controller_id': 1,
 'authorized_operations': ['CREATE',
  'ALTER',
  'DESCRIBE',
  'CLUSTER_ACTION',
  'DESCRIBE_CONFIGS',
  'ALTER_CONFIGS',
  'IDEMPOTENT_WRITE']}

In [30]:
from kafka.admin import NewTopic

In [31]:
# define topic configuration 
sensor_topic = NewTopic(
    name= "sensor-telemetry",
    num_partitions=3,
    replication_factor=1
)

In [32]:
# create  topic in kafka 
admin.create_topics(
    new_topics=[sensor_topic],
    validate_only=False
)

print("topics crreated sucessfully")

TopicAlreadyExistsError: [Error 36] TopicAlreadyExistsError: Request 'CreateTopicsRequest_v3(create_topic_requests=[(topic='sensor-telemetry', num_partitions=3, replication_factor=1, replica_assignment=[], configs=[])], timeout=30000, validate_only=False)' failed with response 'CreateTopicsResponse_v3(throttle_time_ms=0, topic_errors=[(topic='sensor-telemetry', error_code=36, error_message="Topic 'sensor-telemetry' already exists.")])'.

In [33]:
# verify topic exists  or not
topics =admin.list_topics()
print(topics)


['sensor-telemetry']


In [34]:
#inspect topic metadata
from kafka import KafkaConsumer
consumer = KafkaConsumer(
    bootstrap_servers= "localhost:9092"
)

partitions= consumer.partitions_for_topic(
    "sensor-telemetry"
)
print(partitions)

{0, 1, 2}


In [35]:
# import kafka
from kafka import KafkaProducer
import json



In [37]:
#create producer connection
producer= KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer= lambda v: json.dumps(v).encode("utf-8")
)

In [38]:
# create sensor event 

sensor_event= {
    "senesor_id": "S101",
    "temprature": 24.5,
    "humidity": 60
}

print(sensor_event)


{'senesor_id': 'S101', 'temprature': 24.5, 'humidity': 60}


In [40]:
 
# send message to kafka 
future = producer.send(
    "sensor-telemetry",
    value= sensor_event
)

print("message sent")


message sent


In [41]:
# wait for kafka confirmation
record_metadata= future.get(timeout=10)
print(record_metadata)

RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=0, timestamp=1781621064745, checksum=None, serialized_key_size=-1, serialized_value_size=58, serialized_header_size=-1)
